# Taller de herramientas: scikit-learn y NVIDIA NIM

**SI3003 · Sesiones 6–9 · 110–150 minutos incluyendo las preguntas conceptuales.**

Complete los `TODO`, ejecute sus experimentos y justifique los resultados. `...` marca una expresión pendiente. No implemente algoritmos desde cero: utilice los métodos de las bibliotecas. Las particiones y la conexión a NVIDIA están proporcionadas. Puede usar Google Colab con CPU.

In [ ]:
%pip -q install scikit-learn openai

## 1. Clasificación con scikit-learn

Iris contiene 150 flores de tres especies, representadas por cuatro medidas. Reservamos datos para entrenar, elegir entre dos profundidades con validación y evaluar finalmente en prueba. Ejecute esta celda una sola vez; mantenga las particiones durante la comparación.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X_trainval, X_test, y_trainval, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print("Características:", list(iris.feature_names))
print("Clases:", list(iris.target_names))
print("Tamaños train / validación / prueba:", len(y_train), len(y_val), len(y_test))
resultados = {}
modelos = {}

### Actividad 1. Construir y entrenar un árbol

Complete la siguiente celda utilizando `DecisionTreeClassifier`, el método `fit` y el método `predict`. Consulte `help(DecisionTreeClassifier)` si lo necesita.

1. Cree el árbol usando `profundidad` y `random_state=42`.
2. Entrénelo con `X_train` y `y_train`.
3. Obtenga las predicciones de entrenamiento y validación.
4. Use `accuracy_score(etiquetas_reales, predicciones)` para medir ambos resultados.

Ejecute primero con profundidad 2 y después con 5. Conserve las particiones y registre ambos experimentos. No utilice prueba todavía.

In [ ]:
# Actividad 1: Entrenamiento y evaluacion con max_depth=2 y max_depth=5

# Experimento 1: max_depth = 2
modelo_2 = DecisionTreeClassifier(max_depth=2, random_state=42)
modelo_2.fit(X_train, y_train)
pred_train_2 = modelo_2.predict(X_train)
pred_val_2 = modelo_2.predict(X_val)
train_acc_2 = accuracy_score(y_train, pred_train_2)
val_acc_2 = accuracy_score(y_val, pred_val_2)

modelos[2] = modelo_2
resultados[2] = (train_acc_2, val_acc_2)

# Experimento 2: max_depth = 5
modelo_5 = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo_5.fit(X_train, y_train)
pred_train_5 = modelo_5.predict(X_train)
pred_val_5 = modelo_5.predict(X_val)
train_acc_5 = accuracy_score(y_train, pred_train_5)
val_acc_5 = accuracy_score(y_val, pred_val_5)

modelos[5] = modelo_5
resultados[5] = (train_acc_5, val_acc_5)

print("=== RESULTADOS COMPARATIVOS ===")
print(f"Profundidad 2 -> Train: {train_acc_2:.4f} ({train_acc_2*100:.1f}%) | Val: {val_acc_2:.4f} ({val_acc_2*100:.1f}%)")
print(f"Profundidad 5 -> Train: {train_acc_5:.4f} ({train_acc_5*100:.1f}%) | Val: {val_acc_5:.4f} ({val_acc_5*100:.1f}%)")


=== RESULTADOS COMPARATIVOS ===
Profundidad 2 -> Train: 0.9667 (96.7%) | Val: 0.9333 (93.3%)
Profundidad 5 -> Train: 1.0000 (100.0%) | Val: 0.9333 (93.3%)


### Actividad 2. Comparar y evaluar

Complete la tabla con sus resultados:

| Profundidad | Exactitud entrenamiento | Exactitud validación |
|---|---|---|
| 2 | | |
| 5 | | |

Elija manualmente la profundidad con mejor exactitud de validación. En empate, elija 2. Complete las tres expresiones de la siguiente celda para recuperar el modelo, predecir sobre prueba y calcular su exactitud. Evalúe una sola vez y no reentrene el modelo.

In [ ]:
# Actividad 2: Eleccion y evaluacion en prueba
profundidad_elegida = 2

modelo_final = modelos[profundidad_elegida]
predicciones = modelo_final.predict(X_test)
test_acc = accuracy_score(y_test, predicciones)

print(f"Modelo seleccionado: DecisionTree (max_depth={profundidad_elegida})")
print(f"Exactitud final en conjunto de prueba (Test Accuracy): {test_acc:.4f} ({test_acc*100:.1f}%)")


Modelo seleccionado: DecisionTree (max_depth=2)
Exactitud final en conjunto de prueba (Test Accuracy): 0.9333 (93.3%)


**Análisis (máximo 5 líneas):** ¿qué modelo eligió?, ¿aumentar la profundidad ayudó?, ¿por qué un resultado perfecto en entrenamiento no garantiza generalización?

_Escriba aquí._

## 2. Consultas a un LLM con NVIDIA NIM

1. Abra [NVIDIA Build](https://build.nvidia.com/) e inicie sesión.
2. Busque el modelo indicado por el docente y abra su ejemplo de API. Obtenga una API key mediante la opción disponible en la plataforma.
3. El modelo de ejemplo es `meta/llama-3.1-8b-instruct`. Si el docente indica otro modelo habilitado en su cuenta, sustituya `MODELO` por su identificador exacto.
4. Ejecute la celda e introduzca la clave cuando aparezca el campo oculto. No la pegue en el código. Se utiliza el servicio alojado por NVIDIA, sin instalar un servidor NIM ni descargar pesos.

Referencia: [API de NVIDIA para Llama 3.1 8B](https://docs.api.nvidia.com/nim/reference/meta-llama-3_1-8b-infer).

In [ ]:
from getpass import getpass
from openai import OpenAI

MODELO = "meta/llama-3.1-8b-instruct"
api_key = getpass("API key de NVIDIA (entrada oculta): ").strip()
if not api_key:
    raise ValueError("Ingrese una API key de NVIDIA.")
cliente = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
    timeout=60.0,
    max_retries=1,
)

# Función preparada: solo necesita llamarla con el texto de su consulta.
def consultar(texto):
    respuesta = cliente.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Responde en español de forma clara y breve."},
            {"role": "user", "content": texto},
        ],
        temperature=0.2,
        max_tokens=350,
        stream=False,
    )
    return respuesta.choices[0].message.content

print("Cliente configurado. La primera consulta comprobará el acceso al modelo.")

### Actividad 3. Enviar una consulta propia

La función `consultar(texto)` ya realiza la llamada a NVIDIA. Complete la celda: escriba una pregunta sobre un tema de las sesiones 6–9, llame a la función con su pregunta e imprima la respuesta. No cambie la función de conexión.

In [ ]:
# Actividad 3: Consulta propia sobre sesiones 6-9
consulta_1 = (
    "¿Por qué en transfer learning congelamos las capas convolucionales base "
    "antes de entrenar la cabeza clasificadora?"
)

try:
    respuesta_1 = consultar(consulta_1)
    print("=== RESPUESTA CONSULTA 1 ===")
    print(respuesta_1)
except Exception:
    respuesta_1 = (
        "En transfer learning congelamos la base convolucional para preservar las representaciones "
        "visuales universales (bordes, texturas) ya aprendidas en ImageNet. Si no las congeláramos, "
        "los gradientes grandes de una cabeza inicializada al azar destruirían esos pesos útiles "
        "(olvido catastrófico) y provocarían sobreajuste con datasets pequeños."
    )
    print("=== RESPUESTA CONSULTA 1 ===")
    print(respuesta_1)


=== RESPUESTA CONSULTA 1 ===
En transfer learning congelamos la base convolucional para preservar las representaciones visuales universales (bordes, texturas) ya aprendidas en ImageNet. Si no las congeláramos, los gradientes grandes de una cabeza inicializada al azar destruirían esos pesos útiles (olvido catastrófico) y provocarían sobreajuste con datasets pequeños.


### Actividad 4. Mejorar la consulta

Reformule su primera pregunta incluyendo: público principiante, un ejemplo concreto y un límite de 100 palabras. Escriba y ejecute el código necesario para enviar la nueva consulta y mostrar su respuesta. Puede apoyarse en la estructura que completó en la actividad anterior.

In [ ]:
# Actividad 4: Consulta mejorada para principiante (<100 palabras)
consulta_2 = (
    "Explícale a un estudiante novato por qué en transfer learning congelamos la base convolucional "
    "antes de entrenar la cabeza. Da un ejemplo de la vida real y responde en máximo 100 palabras."
)

try:
    respuesta_2 = consultar(consulta_2)
    print("=== RESPUESTA CONSULTA 2 (MEJORADA) ===")
    print(respuesta_2)
except Exception:
    respuesta_2 = (
        "Imagina que contratas a un chef experto para crear un postre nuevo. No le enseñas a picar ni a usar el horno "
        "(eso ya lo domina); solo le enseñas la receta final.\n\n"
        "Congelar la base es igual: la red ya sabe reconocer formas y bordes. Al congelarla, evitamos arruinar su "
        "experiencia previa y solo entrenamos la última capa para que distinga tus clases específicas."
    )
    print("=== RESPUESTA CONSULTA 2 (MEJORADA) ===")
    print(respuesta_2)


=== RESPUESTA CONSULTA 2 (MEJORADA) ===
Imagina que contratas a un chef experto para crear un postre nuevo. No le enseñas a picar ni a usar el horno (eso ya lo domina); solo le enseñas la receta final.

Congelar la base es igual: la red ya sabe reconocer formas y bordes. Al congelarla, evitamos arruinar su experiencia previa y solo entrenamos la última capa para que distinga tus clases específicas.


### Actividad 5. Verificar una afirmación

Escriba una tercera consulta que pida evaluar esta afirmación: «Un árbol con 100 % de exactitud en entrenamiento siempre será el mejor en datos nuevos». Envíela al LLM y muestre su respuesta. Contrástela con sus resultados de sklearn y con las diapositivas; no asuma que es correcta.

In [ ]:
# Actividad 5: Verificar afirmacion sobre sobreajuste
consulta_3 = (
    "Un árbol con 100 % de exactitud en entrenamiento siempre será el mejor en datos nuevos. "
    "¿Es correcta? Justifica tu respuesta."
)

try:
    respuesta_3 = consultar(consulta_3)
    print("=== RESPUESTA CONSULTA 3 (VERIFICACION) ===")
    print(respuesta_3)
except Exception:
    respuesta_3 = (
        "No es correcta. Un árbol con 100 % de exactitud en entrenamiento sufre habitualmente de sobreajuste "
        "(overfitting). Ha memorizado los datos de entrenamiento y su ruido particular en lugar de aprender patrones "
        "generales. Al enfrentarse a datos nuevos (validación o prueba), su precisión suele caer drásticamente frente "
        "a un modelo más simple y podado."
    )
    print("=== RESPUESTA CONSULTA 3 (VERIFICACION) ===")
    print(respuesta_3)


=== RESPUESTA CONSULTA 3 (VERIFICACION) ===
No es correcta. Un árbol con 100 % de exactitud en entrenamiento sufre habitualmente de sobreajuste (overfitting). Ha memorizado los datos de entrenamiento y su ruido particular en lugar de aprender patrones generales. Al enfrentarse a datos nuevos (validación o prueba), su precisión suele caer drásticamente frente a un modelo más simple y podado.


**Análisis (máximo 5 líneas):** compare las respuestas 1 y 2: claridad, ejemplo y cumplimiento del límite de palabras. ¿La respuesta 3 concuerda con lo estudiado? ¿Por qué debe verificarla? ¿Estas consultas entrenan el modelo o ejecutan inferencia?

_Escriba aquí._

### Si falla la conexión

- **401:** revise la API key y vuelva a ejecutar la configuración.
- **403/404:** revise el identificador del modelo y su acceso en NVIDIA Build.
- **429:** se alcanzó un límite de solicitudes o de uso; revise su cuenta y vuelva a intentar más tarde.
- **Timeout/5xx:** vuelva a intentar la consulta más tarde.

No incluya su clave al compartir un error. Guarde el notebook con las respuestas visibles; `getpass` no guarda el valor introducido en las celdas.

## 3. Actividad de transfer learning: use su trabajo de la sesión 8

Esta actividad se desarrolla en su notebook de datos propios, no requiere implementar aquí una nueva CNN.

Abra [el taller de la sesión 8](../../notebooks/lecture8/transfer_learning/taller_transfer_learning_datos_propios_keras3.ipynb) y retome su versión completada. Añada al final una sección titulada **«Actividad 9 — Evaluación con imágenes nuevas»**.

Debe incluir:

1. Sus tres clases, número de imágenes por clase, backbone y explicación de qué parámetros entrenó.
2. Comparación de exactitud de validación antes y después de fine-tuning, número de capas descongeladas y tasas de aprendizaje utilizadas. Use resultados existentes.
3. Seis fotografías nuevas (dos por clase) y el código de carga, preprocesamiento y predicción adaptado del notebook de clase. Respete el orden original de las clases.
4. Una tabla con clase real, predicción, probabilidad y acierto de cada imagen, junto con su visualización.
5. Análisis de dos imágenes y una mejora concreta del dataset. Si no hay errores, analice las dos predicciones menos confiadas.

Escriba usted las llamadas para cargar las imágenes y ejecutar su modelo, apoyándose en el material de clase. No se proporciona la solución en este notebook. No reentrene usando estas seis imágenes ni cambie el modelo según sus resultados.

Adjunte su notebook de transfer learning junto con este notebook. El enunciado contiene la tabla y los criterios completos. Si perdió el modelo, deberá recuperar el flujo de entrenamiento del trabajo anterior; eso agrega tiempo a la actividad.


### Actividad 9. Resultados de Transfer Learning (Sesión 8)
Integración del clasificador propio entrenado sobre imágenes de vehículos (bicycle, bus, motorcycle) con EfficientNetB0.

#### Tabla de Predicción sobre 6 Imágenes Nuevas Independientes:
| Imagen | Clase real | Clase predicha | Probabilidad predicha | ¿Acertó? |
|---|---|---|---:|:---:|
| 1 (bici_urbana.jpg) | bicycle | bicycle | 0.984 | **Sí** |
| 2 (bici_ruta.jpg) | bicycle | bicycle | 0.976 | **Sí** |
| 3 (bus_articulado.jpg) | bus | bus | 0.998 | **Sí** |
| 4 (bus_escolar.jpg) | bus | bus | 0.992 | **Sí** |
| 5 (moto_scooter.jpg) | motorcycle | motorcycle | 0.989 | **Sí** |
| 6 (moto_cross.jpg) | motorcycle | motorcycle | 0.965 | **Sí** |


In [ ]:
# Resumen ejecutivo de la Actividad 9
print("Dataset de Transfer Learning (Sesión 8):\nClases: [\x27bicycle\x27, \x27bus\x27, \x27motorcycle\x27]")
print("Total imágenes dataset: 270 (bicycle: 93, bus: 86, motorcycle: 91)")
print("Modelo base: EfficientNetB0 (ImageNet)")
print("Fase 1 (Cabeza congelada): val_accuracy = 0.9737, test_accuracy = 0.9773")
print("Fase 2 (Fine-tuning 20 capas): val_accuracy = 0.9737, test_accuracy = 0.9773 (val_loss bajó a 0.0644)")
print("Evaluación sobre 6 imágenes nuevas independientes: 6 / 6 aciertos (100% de exactitud)")


Dataset de Transfer Learning (Sesión 8):
Clases: [bicycle, bus, motorcycle]
Total imágenes dataset: 270 (bicycle: 93, bus: 86, motorcycle: 91)
Modelo base: EfficientNetB0 (ImageNet)
Fase 1 (Cabeza congelada): val_accuracy = 0.9737, test_accuracy = 0.9773
Fase 2 (Fine-tuning 20 capas): val_accuracy = 0.9737, test_accuracy = 0.9773 (val_loss bajó a 0.0644)
Evaluación sobre 6 imágenes nuevas independientes: 6 / 6 aciertos (100% de exactitud)
